In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
from scipy.stats import mannwhitneyu
from pyucell import compute_ucell_scores
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import re

# SIGMA vs UCell - Tem cytotoxic T cell extraction
## Dominguez et al. cross-tissue human immune atlas (Science, 2022)

**Comparison of SIGMA and UCell (v2, combined positive/negative signatures)
for the targeted extraction of effector memory cytotoxic T cells.**

### Datasets
- Full dataset: Dominguez Conde et al., Science 2022
  (doi: 10.1126/science.abl5197)
- SIGMA output: merge.h5ad produced by the SIGMA pipeline
  (score threshold >= 0.95)

### UCell reference
- Andreatta & Carmona, Bioinformatics 2026
  (doi: 10.1093/bioinformatics/btag055)

### Validation approach
Selections are evaluated using three biologically unambiguous
marker panels, independent of clustering or annotation quality:
- T cell identity markers (CD3E, TRAC, CD8A, CD8B)
- NK cell contamination markers (NCAM1, KLRD1, FCGR3A, GNLY)
- Non-cytotoxic T cell markers (CD4, FOXP3, TCF7, SELL)

---
## Configuration

In [ ]:
# --- Paths ---
PATH_FULL  = "/home/lemgui01/raw_STUDIES_thesis/healthy/Dominguez_science_science-abl5197.h5ad"
PATH_SIGMA = "/mnt/projects_tn03/AD_EB_singlecell/article/results_dominguez/merge.h5ad"

# --- SIGMA threshold ---
SIGMA_THRESHOLD = 0.95

# --- UCell threshold ---
UCELL_PERCENTILE = 99

# --- Signatures ---
TEM_POSITIVE = ["IL7R", "PRF1", "NKG7", "CX3CR1",
                "GZMB", "GZMK", "CD44"]

TEM_NEGATIVE = ["SELL", "TCF7", "TRGC1", "CXCR5",
                "FOXP3", "LEF1", "TRDC", "CCR7",
                "TRGC2", "CD27"]

TEM_UCELL_COMBINED = ([f"{g}+" for g in TEM_POSITIVE] +
                      [f"{g}-" for g in TEM_NEGATIVE])

# --- Validation marker panels ---
T_CELL_MARKERS = ["CD3E", "TRAC", "CD8A", "CD8B"]
NK_MARKERS = ["NCAM1", "KLRD1", "FCGR3A", "GNLY"]
NON_CYTOTOXIC  = ["CD4", "FOXP3", "TCF7", "SELL"]

# --- Plot colors ---
GREEN = "#2ecc71"   # SIGMA+
ORANGE = "#e67e22"   # UCell+
rng = np.random.default_rng(42)

---
## 1. Data loading

In [ ]:
adata_full  = sc.read_h5ad(PATH_FULL)
adata_sigma = sc.read_h5ad(PATH_SIGMA)

print(f"adata_full  : {adata_full.shape[0]:,} cells x {adata_full.shape[1]:,} genes")
print(f"adata_sigma : {adata_sigma.shape[0]:,} cells x {adata_sigma.shape[1]:,} genes")

---
## 2. SIGMA+ labeling

In [ ]:
adata_sigma_thresh = adata_sigma[adata_sigma.obs["score"] >= SIGMA_THRESHOLD].copy()

adata_sigma_thresh.obs_names = adata_sigma_thresh.obs_names.str.replace(
    r"-Dominguez_.*?_data$", "", regex=True
)

adata_full.obs["sigma_positive"] = adata_full.obs_names.isin(set(adata_sigma_thresh.obs_names))

print(f"SIGMA+ : {adata_full.obs['sigma_positive'].sum():,}")

---
## 3. UCell score computation and thresholding

In [ ]:
compute_ucell_scores(
    adata_full,
    {"tem_combined": TEM_UCELL_COMBINED},
    suffix="_UCell"
)

ucell_threshold = np.percentile(adata_full.obs["tem_combined_UCell"], UCELL_PERCENTILE)

adata_full.obs["ucell_positive"] = adata_full.obs["tem_combined_UCell"] >= ucell_threshold

mask_sigma = adata_full.obs["sigma_positive"]
mask_ucell = adata_full.obs["ucell_positive"]

print(f"UCell threshold ({UCELL_PERCENTILE}th pct) : {ucell_threshold:.4f}")
print(f"SIGMA+ : {mask_sigma.sum():,}")
print(f"UCell+ : {mask_ucell.sum():,}")
print(f"Overlap: {(mask_sigma & mask_ucell).sum():,}")

n_sigma = int(mask_sigma.sum())
n_ucell = int(mask_ucell.sum())
n_both = int((mask_sigma & mask_ucell).sum())

n_sigma_only = n_sigma - n_both
n_ucell_only = n_ucell - n_both

fig, ax = plt.subplots(figsize=(7, 6))

c1 = plt.Circle((0.43, 0.5), 0.28, color=GREEN, alpha=0.45, ec="black", lw=1.2)
c2 = plt.Circle((0.57, 0.5), 0.28, color=ORANGE, alpha=0.45, ec="black", lw=1.2)
ax.add_patch(c1)
ax.add_patch(c2)

ax.text(0.235, 0.50, f"{n_sigma_only:,}", ha="center", va="center",
    fontsize=14, fontweight="bold")
ax.text(0.50, 0.50, f"{n_both:,}", ha="center", va="center",
    fontsize=14, fontweight="bold")
ax.text(0.775, 0.50, f"{n_ucell_only:,}", ha="center", va="center",
    fontsize=14, fontweight="bold")

ax.text(0.30, 0.78, "SIGMA+", ha="center", va="center",
    fontsize=11, fontweight="bold", color=GREEN)
ax.text(0.70, 0.78, "UCell+", ha="center", va="center",
    fontsize=11, fontweight="bold", color=ORANGE)

ax.set_title("Venn diagram: SIGMA+ vs UCell+", fontsize=13, fontweight="bold")
ax.set_xlim(0.1, 0.9)
ax.set_ylim(0.15, 0.9)
ax.set_aspect("equal")
ax.axis("off")

plt.tight_layout()
fig.savefig("figures/SUP2-A.svg", bbox_inches="tight", dpi=300)
plt.show()

---
## 4. log1p-CPM normalization

In [ ]:
adata_norm = adata_full.copy()
sc.pp.normalize_total(adata_norm, target_sum=1e6)
sc.pp.log1p(adata_norm)

---
## 5. Utility functions

In [ ]:
def get_expression(adata, mask, gene):
    """
    Retrieve log1p-CPM expression values for a given gene and mask.
    """
    if gene not in adata.var_names:
        return np.array([])
    idx = adata.var_names.get_loc(gene)
    expr = adata.X[mask, idx]
    if hasattr(expr, "toarray"):
        expr = expr.toarray().flatten()
    return np.array(expr).flatten()

def mean_panel(adata, mask, genes):
    """
    Compute the mean expression of a list of genes for a given mask.
    """
    genes_ok = [g for g in genes if g in adata.var_names]
    if not genes_ok:
        return np.zeros(int(mask.sum()))
    idx = [adata.var_names.get_loc(g) for g in genes_ok]
    expr = adata.X[mask][:, idx]
    if hasattr(expr, "toarray"):
        expr = expr.toarray()
    return np.mean(expr, axis=1)

def format_pvalue(pval):
    return f"{pval:.3e}"

def boxplot_panel(ax, v1, v2, title, rng, n_sample=1000):
    """
    Create a boxplot comparing two sets of expression values (v1 and v2) on the given axis.
    """
    for i, (vals, color) in enumerate(
        [(v1, GREEN), (v2, ORANGE)], start=1
    ):
        bp = ax.boxplot(
            [v1, v2][i - 1:i],
            positions=[i], patch_artist=True, widths=0.5,
            notch=False, showfliers=False,
            medianprops=dict(color="black", linewidth=2), zorder=3
        )
        bp["boxes"][0].set_facecolor(color)
        bp["boxes"][0].set_alpha(0.75)
        for w in bp["whiskers"]: w.set(linewidth=1.0, linestyle="--")
        for c in bp["caps"]:     c.set(linewidth=1.0)

        s   = rng.choice(len(vals), size=min(n_sample, len(vals)),
                         replace=False)
        jit = rng.uniform(-0.12, 0.12, size=len(s))
        ax.scatter(i + jit, vals[s], color=color,
                   alpha=0.3, s=6, zorder=2, linewidths=0)

    stat, pval = mannwhitneyu(v1, v2, alternative="two-sided")
    ax.set_title(title + f"\np = {format_pvalue(pval)}", fontsize=11, fontweight="bold")
    ax.set_xticks([1, 2])
    ax.set_xticklabels(
        [f"SIGMA+\n(n={len(v1):,})", f"UCell+\n(n={len(v2):,})"],
        fontsize=9
    )
    ax.set_ylabel("log1p-CPM", fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return pval

def plot_gene_panel(genes, suptitle, figname, rng, n_cols=4, n_sample=1000):
    """
    Plot expression levels for a list of genes across two conditions.
    """
    genes_ok = [g for g in genes if g in adata_norm.var_names]
    n_rows   = int(np.ceil(len(genes_ok) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
    axes = axes.flatten()
    for idx, gene in enumerate(genes_ok):
        boxplot_panel(axes[idx],
                      get_expression(adata_norm, mask_sigma.values, gene),
                      get_expression(adata_norm, mask_ucell.values, gene),
                      gene, rng, n_sample)
    for idx in range(len(genes_ok), len(axes)):
        axes[idx].axis("off")
    fig.suptitle(suptitle, fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{figname}.svg", bbox_inches="tight", dpi=300)
    plt.show()

---
## 6. Figure C - T cell identity markers

In [ ]:
plot_gene_panel(
    T_CELL_MARKERS,
    "T cell identity markers | SIGMA+ vs UCell+",
    "figures/SUP2-C",
    rng
)

---
## 7. Figure D - NK cell contamination markers

In [ ]:
plot_gene_panel(
    NK_MARKERS,
    "NK cell contamination markers | SIGMA+ vs UCell+",
    "figures/SUP2-D",
    rng
)

---
## 8. Figure E - Non-cytotoxic T cell markers

In [ ]:
plot_gene_panel(
    NON_CYTOTOXIC,
    "Non-cytotoxic T cell markers | SIGMA+ vs UCell+",
    "figures/SUP2-E",
    rng
)

---
## 9. Figure B - Summary panel scores

In [ ]:
panels = [
    (T_CELL_MARKERS, "T cell identity markers"),
    (NK_MARKERS, "NK contamination markers"),
    (NON_CYTOTOXIC,  "Non-cytotoxic T cell markers"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
rows = []

for ax, (genes, title) in zip(axes, panels):
    s = mean_panel(adata_norm, mask_sigma.values, genes)
    u = mean_panel(adata_norm, mask_ucell.values, genes)
    pval = boxplot_panel(ax, s, u, title, rng, n_sample=500)
    ax.set_ylabel("Mean log1p-CPM", fontsize=10)
    rows.append({"Panel": title.replace("\n", " "),
                 "SIGMA+ median": round(float(np.median(s)), 3),
                 "UCell+ median": round(float(np.median(u)), 3),
                 "p-value": format_pvalue(pval)})

fig.suptitle("Biological summary | SIGMA+ vs UCell+",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig("figures/SUP2-B.svg", bbox_inches="tight", dpi=300)
plt.show()

---
## 10. Final figure

In [ ]:
fig = plt.figure(figsize=(20, 20))
gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.5, wspace=0.45)

ax_venn = fig.add_subplot(gs[0, 0])
axB = [fig.add_subplot(gs[0, i]) for i in range(1, 4)]
axC = [fig.add_subplot(gs[1, i]) for i in range(4)]
axD = [fig.add_subplot(gs[2, i]) for i in range(4)]
axE = [fig.add_subplot(gs[3, i]) for i in range(4)]

# A
c1 = plt.Circle((0.43, 0.5), 0.28, color=GREEN, alpha=0.45, ec="black", lw=1.2)
c2 = plt.Circle((0.57, 0.5), 0.28, color=ORANGE, alpha=0.45, ec="black", lw=1.2)
ax_venn.add_patch(c1)
ax_venn.add_patch(c2)
ax_venn.text(0.235, 0.50, f"{n_sigma_only:,}", ha="center", va="center", fontsize=11, fontweight="bold")
ax_venn.text(0.50, 0.50, f"{n_both:,}", ha="center", va="center", fontsize=11, fontweight="bold")
ax_venn.text(0.775, 0.50, f"{n_ucell_only:,}", ha="center", va="center", fontsize=11, fontweight="bold")
ax_venn.text(0.30, 0.78, "SIGMA+", ha="center", fontsize=11, fontweight="bold", color=GREEN)
ax_venn.text(0.70, 0.78, "UCell+", ha="center", fontsize=11, fontweight="bold", color=ORANGE)
ax_venn.set_xlim(0.1, 0.9); ax_venn.set_ylim(0.15, 0.9)
ax_venn.set_aspect("equal"); ax_venn.axis("off")
ax_venn.text(-0.12, 1.15, "A", transform=ax_venn.transAxes, fontsize=28, fontweight="bold", va="top")

# B
panels_B = [
    (T_CELL_MARKERS, "T cell identity markers"),
    (NK_MARKERS, "NK contamination markers"),
    (NON_CYTOTOXIC,  "Non-cytotoxic T cell markers"),
]
rows = []
for ax, (genes, title) in zip(axB, panels_B):
    s = mean_panel(adata_norm, mask_sigma.values, genes)
    u = mean_panel(adata_norm, mask_ucell.values, genes)
    pval = boxplot_panel(ax, s, u, title, rng, n_sample=500)
    ax.set_ylabel("Mean log1p-CPM", fontsize=9)
    rows.append({"Panel": title,
                 "SIGMA+ median": round(float(np.median(s)), 3),
                 "UCell+ median": round(float(np.median(u)), 3),
                 "p-value": format_pvalue(pval)})
axB[0].text(-0.12, 1.15, "B", transform=axB[0].transAxes, fontsize=28, fontweight="bold", va="top")

# C, D, E
for axrow, genes_list, letter, title in [
    (axC, T_CELL_MARKERS, "C", "T cell identity markers | SIGMA+ vs UCell+"),
    (axD, NK_MARKERS, "D", "NK cell contamination markers | SIGMA+ vs UCell+"),
    (axE, NON_CYTOTOXIC,  "E", "Non-cytotoxic T cell markers | SIGMA+ vs UCell+"),
]:
    genes_ok = [g for g in genes_list if g in adata_norm.var_names]
    for ax, gene in zip(axrow, genes_ok):
        boxplot_panel(ax,
                      get_expression(adata_norm, mask_sigma.values, gene),
                      get_expression(adata_norm, mask_ucell.values, gene),
                      gene, rng)
    for ax in axrow[len(genes_ok):]:
        ax.axis("off")
    axrow[0].text(-0.12, 1.15, letter, transform=axrow[0].transAxes,
                  fontsize=28, fontweight="bold", va="top")
    top = axrow[0].get_position().y1
    fig.text(0.5, top + 0.025, title, ha="center", va="bottom",
             fontsize=12, fontweight="bold")
    
fig.savefig("figures/SUP2-full.svg", bbox_inches="tight", dpi=300)
plt.show()